# Device theft and tampering  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 2.2.c: Device theft/tampering incident triage

**Skill:** Analyze Risk · **Maps to:** 2.2.B, 2.2.C · **Language:** Python 3 (pytest)

## Problem
The IT security team receives reports whenever a device is reported stolen or found to show signs
of tampering. Each report needs to be triaged into a risk level so the team knows what needs an
immediate incident response versus what can be logged and monitored.

Your task is to write a function `solve(data)` that takes a list of incident reports and returns a
dictionary mapping each incident's `id` to a risk level: `"LOW"`, `"MEDIUM"`, or `"HIGH"`.

### Rules, by incident `type`

**`type == "theft"`** (a device was physically taken):
1. `"HIGH"` if the device was **not** encrypted (`encrypted` is `False`), an unencrypted stolen
   device exposes its data with no cryptographic barrier.
2. `"MEDIUM"` if the device **was** encrypted, but `contains_sensitive_data` is `True` **and**
   `hours_since_check` is greater than `48`, encryption protects the data, but a long detection
   delay on a sensitive device still raises risk (e.g., the device may have been unlocked and
   in active use when taken).
3. `"LOW"` otherwise.

**`type == "tampering"`** (a device was found to show signs of covert alteration):
1. `"HIGH"` if the tamper-evident seal was **not** intact (`seal_intact` is `False`), physical
   evidence the enclosure was opened.
2. `"MEDIUM"` if the seal **was** intact but `hours_since_check` is greater than `48`, a long
   window since the last inventory check means tampering that does not require opening the
   enclosure (e.g., a rogue USB implant) could have gone unnoticed even with an intact seal.
3. `"LOW"` otherwise.

Any other `type` value should be treated as `"LOW"`.

## Function contract
Implement `solve(data: list[dict]) -> dict[str, str]` in the code editor.

## Input
A list of dictionaries, each representing one incident report:
```python
[
    {
        "id": "inc-1",
        "type": "theft",
        "encrypted": False,
        "seal_intact": True,
        "contains_sensitive_data": True,
        "hours_since_check": 10,
    },
    {
        "id": "inc-2",
        "type": "tampering",
        "encrypted": True,
        "seal_intact": False,
        "contains_sensitive_data": False,
        "hours_since_check": 5,
    },
]
```
Every report includes all five fields (`type`, `encrypted`, `seal_intact`,
`contains_sensitive_data`, `hours_since_check`), even if a field is not relevant to that report's
`type` (e.g., `seal_intact` is present but ignored for `type == "theft"`).

## Output
A dictionary mapping each incident's `"id"` to its risk level string:
```python
{
    "inc-1": "HIGH",
    "inc-2": "HIGH",
}
```

## Constraints
- `type` is `"theft"` or `"tampering"` (or, in hidden tests, possibly another string that must
  default to `"LOW"`).
- `encrypted`, `seal_intact`, `contains_sensitive_data` are booleans.
- `hours_since_check` is a non-negative number (int or float).
- Pure function, deterministic, standard library only.

## How to run
```bash
Your code passes when every test below shows a checkmark (some are hidden until you run them).
```

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(data: list[dict]) -> dict[str, str]
```

### Input (`data`)
A list of dictionaries representing device incident reports. Each dictionary contains:
- `id` (str): Unique identifier for the incident.
- `type` (str): `"theft"` or `"tampering"` (other values default to `"LOW"`).
- `encrypted` (bool): Whether the device's storage was encrypted.
- `seal_intact` (bool): Whether the tamper-evident seal was intact when discovered.
- `contains_sensitive_data` (bool): Whether the device held sensitive data.
- `hours_since_check` (float/int): Hours since the device was last verified present/unaltered.

### Output (`result`)
A dictionary where keys are the `id` of each incident from the input and values are the risk
level: `"LOW"`, `"MEDIUM"`, or `"HIGH"`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- Rules by `type` (see the problem statement above for full detail):
  1. `theft`: not `encrypted` -> `HIGH`; `encrypted` and `contains_sensitive_data` and
     `hours_since_check > 48` -> `MEDIUM`; else `LOW`.
  2. `tampering`: not `seal_intact` -> `HIGH`; `seal_intact` and `hours_since_check > 48` ->
     `MEDIUM`; else `LOW`.
  3. any other `type` -> `LOW`.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 2.2.c. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""


def _theft_risk(incident: dict) -> str:
    if not incident["encrypted"]:
        return "HIGH"
    if incident["contains_sensitive_data"] and incident["hours_since_check"] > 48:
        return "MEDIUM"
    return "LOW"


def _tampering_risk(incident: dict) -> str:
    if not incident["seal_intact"]:
        return "HIGH"
    if incident["hours_since_check"] > 48:
        return "MEDIUM"
    return "LOW"


def solve(data: list[dict]) -> dict[str, str]:
    results = {}
    for incident in data:
        rid = incident["id"]
        itype = incident["type"]
        if itype == "theft":
            results[rid] = _theft_risk(incident)
        elif itype == "tampering":
            results[rid] = _tampering_risk(incident)
        else:
            results[rid] = "LOW"
    return results


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def _mk(id_, type_, encrypted, seal_intact, sensitive, hours):
    return {
        "id": id_,
        "type": type_,
        "encrypted": encrypted,
        "seal_intact": seal_intact,
        "contains_sensitive_data": sensitive,
        "hours_since_check": hours,
    }


# ---- visible cases -------------------------------------------------------
def test_theft_unencrypted_is_high():
    data = [_mk("i1", "theft", False, True, False, 1)]
    assert impl.solve(data) == {"i1": "HIGH"}


def test_theft_encrypted_sensitive_long_delay_is_medium():
    data = [_mk("i1", "theft", True, True, True, 50)]
    assert impl.solve(data) == {"i1": "MEDIUM"}


def test_theft_encrypted_sensitive_short_delay_is_low():
    data = [_mk("i1", "theft", True, True, True, 10)]
    assert impl.solve(data) == {"i1": "LOW"}


def test_theft_encrypted_not_sensitive_is_low():
    data = [_mk("i1", "theft", True, True, False, 100)]
    assert impl.solve(data) == {"i1": "LOW"}


def test_tampering_seal_broken_is_high():
    data = [_mk("i1", "tampering", True, False, False, 1)]
    assert impl.solve(data) == {"i1": "HIGH"}


def test_tampering_seal_intact_long_delay_is_medium():
    data = [_mk("i1", "tampering", True, True, False, 72)]
    assert impl.solve(data) == {"i1": "MEDIUM"}


def test_tampering_seal_intact_short_delay_is_low():
    data = [_mk("i1", "tampering", True, True, False, 5)]
    assert impl.solve(data) == {"i1": "LOW"}


def test_unknown_type_is_low():
    data = [_mk("i1", "lost", False, False, True, 1000)]
    assert impl.solve(data) == {"i1": "LOW"}


# ---- hidden / edge cases -------------------------------------------------
def test_edge_empty():
    assert impl.solve([]) == {}


def test_boundary_48_hours_is_not_yet_escalated():
    # hours_since_check == 48 is NOT > 48, so should stay LOW/at baseline.
    data = [
        _mk("theft48", "theft", True, True, True, 48),
        _mk("tamper48", "tampering", True, True, False, 48),
    ]
    assert impl.solve(data) == {"theft48": "LOW", "tamper48": "LOW"}


def test_multiple_mixed():
    data = [
        _mk("a", "theft", False, True, True, 1),
        _mk("b", "theft", True, True, True, 60),
        _mk("c", "tampering", True, False, False, 1),
        _mk("d", "tampering", True, True, False, 60),
        _mk("e", "theft", True, False, False, 1),
    ]
    expected = {
        "a": "HIGH",
        "b": "MEDIUM",
        "c": "HIGH",
        "d": "MEDIUM",
        "e": "LOW",
    }
    assert impl.solve(data) == expected


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
